# 3. 모델링

다양한 분류 알고리즘을 적용하고 성능을 비교합니다.
- 기본 모델: RandomForest, LightGBM, XGBoost
- 하이퍼파라미터 튜닝: Ray Tune + Optuna
- AutoML: AutoGluon (Stacking + Bagging)
- 앙상블: Soft Voting

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

train_data = pd.read_csv('/content/data/train_processed.csv')
test_raw = pd.read_csv('/content/data/test_processed.csv')
test_ids = pd.read_csv('/content/data/test_ids.csv')['ID']

X = train_data.drop('label', axis=1)
y = train_data['label']
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

Features: 45, Samples: 7000


## 3.1 기본 모델 비교 (Cross Validation)

In [2]:
import lightgbm as lgb
import xgboost as xgb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'LightGBM': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss'),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name}: {scores.mean():.5f} (±{scores.std():.5f})")

RandomForest: 0.73386 (±0.00534)
LightGBM: 0.72829 (±0.01035)
XGBoost: 0.72729 (±0.01165)


## 3.2 하이퍼파라미터 튜닝 (Ray Tune + Optuna)

LightGBM과 XGBoost에 대해 Ray Tune 기반 하이퍼파라미터 탐색을 수행합니다.

In [4]:
! pip install ray[tune]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 11.8 MB/s eta 0:00:00


In [3]:
import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

train_data_ref = ray.put(X)
target_data_ref = ray.put(y)

2026-04-01 14:24:52,941	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


In [10]:
import sys
!{sys.executable} -m pip install optuna

In [4]:
import optuna
print(optuna.__version__)

4.8.0


In [5]:
import numpy as np
import lightgbm as lgb
import ray

from ray import tune
from ray.air import session
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

train_data_ref = ray.put(X)
target_data_ref = ray.put(y)

def train_lgbm_tune(config):
    try:
        X_train = ray.get(train_data_ref)
        y_train = ray.get(target_data_ref)

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []

        for tr_idx, val_idx in skf.split(X_train, y_train):
            model = lgb.LGBMClassifier(**config, random_state=42, verbose=-1)
            model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            pred = model.predict(X_train.iloc[val_idx])
            scores.append(accuracy_score(y_train.iloc[val_idx], pred))

        session.report({"accuracy": float(np.mean(scores))})

    except Exception as e:
        print("Trial failed:", repr(e))
        raise

lgbm_space = {
    'n_estimators': tune.randint(100, 800),
    'learning_rate': tune.loguniform(1e-3, 0.1),
    'num_leaves': tune.randint(20, 100),
    'max_depth': tune.randint(3, 10),
    'min_child_samples': tune.randint(10, 50),
    'subsample': tune.uniform(0.6, 1.0),
    'colsample_bytree': tune.uniform(0.6, 1.0),
    'reg_alpha': tune.loguniform(1e-3, 1.0),
    'reg_lambda': tune.loguniform(1e-3, 1.0),
}

analysis_lgbm = tune.run(
    train_lgbm_tune,
    config=lgbm_space,
    num_samples=5,
    search_alg=OptunaSearch(metric="accuracy", mode="max"),
    scheduler=ASHAScheduler(metric="accuracy", mode="max"),
    verbose=2,
    fail_fast=False,
)

best_config = analysis_lgbm.get_best_config(metric="accuracy", mode="max")
best_trial = analysis_lgbm.get_best_trial(metric="accuracy", mode="max")

print(f"Best LightGBM Accuracy: {best_trial.last_result['accuracy']:.5f}")
print(f"Best Config: {best_config}")

2026-04-01 14:26:46,255	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
[I 2026-04-01 14:26:50,273] A new study created in memory with name: optuna


+------------------------------------------------------------------------+
| Configuration for experiment     train_lgbm_tune_2026-04-01_14-26-50   |
+------------------------------------------------------------------------+
| Search algorithm                 SearchGenerator                       |
| Scheduler                        AsyncHyperBandScheduler               |
| Number of trials                 5                                     |
+------------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_lgbm_tune_2026-04-01_14-26-50
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-04-01_14-26-29_635509_68951/artifacts/2026-04-01_14-26-50/train_lgbm_tune_2026-04-01_14-26-50/driver_artifacts`

Trial status: 1 PENDING
Current time: 2026-04-01 14:27:01. Total running time: 0s
Logical resource usage: 0/2 CPUs, 0/0 GPUs
+------------------------------------------------------------

2026-04-01 14:28:00,745	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_lgbm_tune_2026-04-01_14-26-50' in 0.0096s.



Trial train_lgbm_tune_d0ec8071 finished iteration 1 at 2026-04-01 14:28:00. Total running time: 58s
+--------------------------------------------------+
| Trial train_lgbm_tune_d0ec8071 result            |
+--------------------------------------------------+
| checkpoint_dir_name                              |
| time_this_iter_s                          4.3425 |
| time_total_s                              4.3425 |
| training_iteration                             1 |
| accuracy                                   0.731 |
+--------------------------------------------------+

Trial train_lgbm_tune_d0ec8071 completed after 1 iterations at 2026-04-01 14:28:00. Total running time: 58s

Trial status: 5 TERMINATED
Current time: 2026-04-01 14:28:00. Total running time: 58s
Logical resource usage: 1.0/2 CPUs, 0/0 GPUs
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 3.3 AutoGluon (AutoML)

AutoGluon의 `best_quality` 프리셋으로 자동 스태킹 + 배깅을 적용합니다.  
"왜 이 모델이 더 잘 동작하는가?" → AutoGluon은 다양한 모델을 자동으로 앙상블하여 단일 모델 대비 일반화 성능이 높습니다.

In [12]:
! pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [7]:
import torch
from autogluon.tabular import TabularPredictor

USE_GPU = torch.cuda.is_available()
print(f"GPU 사용: {USE_GPU}")

save_path = '../outputs/ag_models'

predictor = TabularPredictor(
    label='label',
    eval_metric='accuracy',
    path=save_path,
    problem_type='binary'
).fit(
    train_data,
    presets='best_quality',
    num_bag_folds=10,
    time_limit=600,
    num_gpus=1 if USE_GPU else 0,
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       9.43 GB / 12.67 GB (74.5%)
Disk Space Avail:   74.63 GB / 107.72 GB (69.3%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=10, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` 

GPU 사용: False


(_dystack pid=72344) 	0.7433	 = Validation score   (accuracy)
(_dystack pid=72344) 	52.18s	 = Training   runtime
(_dystack pid=72344) 	0.21s	 = Validation runtime
(_dystack pid=72344) Fitting model: LightGBM_BAG_L1 ... Training model for up to 1139.15s of the 1739.34s of remaining time.
(_dystack pid=72344) 	Fitting 10 child models (S1F1 - S1F10) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.35%)
(_dystack pid=73079) Running DyStack sub-fit ...
(_dystack pid=73079) Beginning AutoGluon training ... Time limit = 150s
(_dystack pid=73079) AutoGluon will save models to "/outputs/ag_models/ds_sub_fit/sub_fit_ho"
(_dystack pid=73079) Train Data Rows:    6222
(_dystack pid=73079) Train Data Columns: 45
(_dystack pid=73079) Label Column:       label
(_dystack pid=73079) Problem Type:       binary
(_dystack pid=73079) Preprocessing data ...
(_dystack pid=73079) Selected class <--> label mapping:  class 1 = 1, class 0 = 0
(_dystack pid=73079) Using Fea

## 3.4 모델 리더보드 확인 및 Soft Voting 앙상블

In [8]:
lb = predictor.leaderboard(train_data, silent=True)
lb_sorted = lb.sort_values(by='score_val', ascending=False)
base_models = lb_sorted[~lb_sorted['model'].str.contains('WeightedEnsemble')]

top_models = base_models['model'].head(10).tolist()
print("Top 10 모델:")
print(base_models[['model', 'score_val']].head(10))

Top 10 모델:
                     model  score_val
1        LightGBMXT_BAG_L2   0.754143
4          LightGBM_BAG_L1   0.743286
5        LightGBMXT_BAG_L1   0.741857
0  RandomForestGini_BAG_L1   0.737571


In [10]:
# 상위 3개 모델로 Soft Voting 앙상블
selected_models = top_models[:3]
print(f"앙상블 모델: {selected_models}")

preds = []
for model in selected_models:
    pred_proba = predictor.predict_proba(test_raw, model=model).iloc[:, 1]
    preds.append(pred_proba)

final_proba = sum(preds) / len(preds)

submission = pd.DataFrame({
    'ID': test_ids,
    'label': (final_proba >= 0.5).astype(int)
})
submission.to_csv('/content/data/result_submission.csv', index=False)
print(f"제출 파일 저장 완료: {submission.shape}")
submission.head()

앙상블 모델: ['LightGBMXT_BAG_L2', 'LightGBM_BAG_L1', 'LightGBMXT_BAG_L1']
제출 파일 저장 완료: (3000, 2)


,ID,label
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,1
3,TEST_0003,1
4,TEST_0004,0
